# HAKE-MER — Eval supplements (cardinality + lexicon ablation)

**PFE — Mariem El Wedani** · SIIVA · Encadrant académique : Sahbi Bahroun · 2025–2026


**No new hyperparameter search.** This notebook only:

1. Ensures DistilBERT checkpoints exist under `runs/` for Step~0, +M1, +M1+M2, M3 NRC, M3 SenticNet (trains missing stacks with the usual 4-epoch protocol).
2. Runs `./run_eval_supplements.sh` (test-only forward passes: `by_gold_cardinality`, M3 lexique zero/shuffle).
3. Downloads a zip of updated `metrics.json` + patched campaign JSON + LaTeX snippets.

**Référence :** comparer les métriques test au chapitre 4 du rapport (même protocole : batch 16, LR 5e-5, 4 époques, trois graines).


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

import os
REPO = os.environ.get("GITHUB_REPO", "khalef-khalil/marii")
WORKDIR = Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0 --device cuda"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
from pathlib import Path

def need_train(glob_pat: str, n: int = 3) -> bool:
    ck = sorted(Path("runs").glob(glob_pat))
    print(glob_pat, len(ck), "checkpoints")
    return len(ck) < n

TRAIN_PLAN = [
    ("distilbert_base_uncased_seed*_baseline_plm/best_model.pt", "./run_baseline_plm_campaign.sh --backbone distilbert-base-uncased"),
    ("distilbert_base_uncased_seed*_m1/best_model.pt", "./run_m1_campaign.sh --backbone distilbert-base-uncased"),
    ("distilbert_base_uncased_seed*_m1_m2/best_model.pt", "./run_m1_m2_campaign.sh --backbone distilbert-base-uncased"),
    ("distilbert_base_uncased_seed*_m1_m2_m3_nrc/best_model.pt", "./run_m1_m2_m3_campaign.sh --lexicon nrc --backbone distilbert-base-uncased"),
    ("distilbert_base_uncased_seed*_m1_m2_m3_senticnet/best_model.pt", "./run_m1_m2_m3_campaign.sh --lexicon senticnet --backbone distilbert-base-uncased"),
]
for glob_pat, cmd in TRAIN_PLAN:
    if need_train(glob_pat):
        print("Training:", cmd)
        get_ipython().system(f"{cmd} {TRAIN_FLAGS}")

In [ ]:
!ALLOW_EVAL=1 DEVICE=cuda ./run_eval_supplements.sh

In [ ]:
from pathlib import Path
for snippet in ["cardinality_snippet.tex", "lexicon_ablation_snippet.tex"]:
    p = Path("reference/artifacts") / snippet
    print("---", snippet, "---")
    print(p.read_text(encoding="utf-8") if p.is_file() else "(missing)")

In [ ]:
import zipfile
from google.colab import files
from pathlib import Path

out_name = "eval_supplements_distilbert.zip"
zip_path = Path(f"/content/{out_name}")
campaigns = [
    "baseline_plm_distilbert_base_uncased_campaign.json",
    "m1_distilbert_base_uncased_campaign.json",
    "m1_m2_distilbert_base_uncased_campaign.json",
    "m1_m2_m3_nrc_distilbert_base_uncased_campaign.json",
    "m1_m2_m3_senticnet_distilbert_base_uncased_campaign.json",
]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in campaigns:
        p = Path("reference/artifacts") / name
        if p.is_file():
            zf.write(p, f"campaigns/{name}")
    for snippet in ("cardinality_snippet.tex", "lexicon_ablation_snippet.tex"):
        p = Path("reference/artifacts") / snippet
        if p.is_file():
            zf.write(p, snippet)
    for m in sorted(Path("runs").glob("distilbert_base_uncased_seed*/metrics.json")):
        zf.write(m, f"metrics/{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

**PFE — Mariem El Wedani** · SIIVA · Encadrant académique : Sahbi Bahroun · 2025–2026

## Integrity

1. **File → Download → .ipynb** → `reference/training_records/step_eval_supplements/colab/`
2. Télécharger le zip et le notebook exécuté ; les conserver pour reproductibilité.